In [7]:
import os
import time
import logging
# from mapbox import Geocoder
import pandas as pd
from shapely.geometry import Point
from tqdm.auto import tqdm
import geopandas as gpd
import numpy as np
from scipy import stats
import osmnx as ox
import geopandas as gpd
import numpy as np

In [8]:
folder_path = "data"

with os.scandir(folder_path) as entries:
    print("Содержимое папки:")
    for entry in entries:
        print(entry.name)

Содержимое папки:
builds.gpkg
fitness_w_centerd.csv
myhouse_RU-MOW_polygons.cpg
myhouse_RU-MOW_polygons.dbf
myhouse_RU-MOW_polygons.prj
myhouse_RU-MOW_polygons.qix
myhouse_RU-MOW_polygons.shp
myhouse_RU-MOW_polygons.shx
new_builds.gpkg
order_boundary.gpkg


### Считываем файлы, смотрим содержимое

In [9]:
builds = gpd.read_file('data/builds.gpkg')
myhouse_polygons = gpd.read_file('data/myhouse_RU-MOW_polygons.shp')
new_builds = gpd.read_file('data/new_builds.gpkg')
boundary = gpd.read_file('data/order_boundary.gpkg')

In [82]:
int(builds['INHAB_count'].sum()/len(builds))

277

In [10]:
new_builds['data.objReady100PercDt'].sort_values()

137     2015-05-07
1736    2015-08-31
1675    2015-12-30
899     2016-02-16
896     2016-02-16
           ...    
428     2029-12-31
431     2029-12-31
426     2029-12-31
4             None
5             None
Name: data.objReady100PercDt, Length: 2067, dtype: object

In [11]:
new_builds.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 2067 entries, 0 to 2066
Data columns (total 15 columns):
 #   Column                          Non-Null Count  Dtype   
---  ------                          --------------  -----   
 0   просто                          2067 non-null   bool    
 1   data.id                         2067 non-null   float64 
 2   data.developer.devId            2067 non-null   float64 
 3   data.developer.devShortCleanNm  2067 non-null   object  
 4   data.region                     2067 non-null   object  
 5   data.address                    2067 non-null   object  
 6   data.objElemLivingCnt           2067 non-null   object  
 7   data.objReady100PercDt          2065 non-null   object  
 8   data.objLkFinishTypeCount       2067 non-null   object  
 9   data.objElemParkingCnt          2067 non-null   object  
 10  data.objSquareLiving            2067 non-null   object  
 11  data.objLkLatitude              2067 non-null   float64 
 12  data.objLkLo

In [12]:
new_builds['data.objReady100PercDt'].sort_values()

137     2015-05-07
1736    2015-08-31
1675    2015-12-30
899     2016-02-16
896     2016-02-16
           ...    
428     2029-12-31
431     2029-12-31
426     2029-12-31
4             None
5             None
Name: data.objReady100PercDt, Length: 2067, dtype: object

In [13]:
builds.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 34375 entries, 0 to 34374
Data columns (total 51 columns):
 #   Column           Non-Null Count  Dtype   
---  ------           --------------  -----   
 0   LAT              34375 non-null  float64 
 1   LON              34375 non-null  float64 
 2   HOUSE_ID         34375 non-null  int64   
 3   ADDRESS          34375 non-null  object  
 4   FIAS             34375 non-null  int64   
 5   YEAR_BLD         31454 non-null  float64 
 6   YEAR_EXPL        33144 non-null  float64 
 7   SERIE            32255 non-null  object  
 8   HOUSE_TYPE       34299 non-null  object  
 9   CAPFOND          33414 non-null  object  
 10  MGMT             32598 non-null  object  
 11  MGMT_LINK        32598 non-null  object  
 12  AVAR             0 non-null      float64 
 13  LEVELS           33222 non-null  float64 
 14  DOORS            32095 non-null  object  
 15  LIFTS            21705 non-null  float64 
 16  RMC              31558 non-null 

### Обрезаем по границам Москвы

In [14]:
# builds
if builds.crs != boundary.crs:
    builds = builds.to_crs(boundary.crs)
builds_clipped = gpd.clip(builds, boundary)

# new builds
if new_builds.crs != boundary.crs:
    new_builds = new_builds.to_crs(boundary.crs)
new_builds_clipped = gpd.clip(new_builds, boundary)

In [15]:
num1 = len(builds_clipped['INHAB'].dropna())/len(builds_clipped)*100
num2 = len(builds_clipped[builds_clipped['INHAB'].isna()].dropna(subset=['RMC_LIVE', 'AREA_LIVE']))/len(builds_clipped)*100
print(f"Доля известного населения: {num1:.2f} %")
print(f"Доля домов без прописанного населения, но с количеством квартир и/или жилой площадью: {num2:.2f} %")
print(f"Доля домов без прописанного населения и без известного числа квартир и/или жилой площади: {100-num1-num2:.2f} %")

Доля известного населения: 78.85 %
Доля домов без прописанного населения, но с количеством квартир и/или жилой площадью: 16.49 %
Доля домов без прописанного населения и без известного числа квартир и/или жилой площади: 4.66 %


### Обрабатываем данные по населению

Рассчитывается численность населения в зданиях на основе различных параметров (количество квартир, жилая площадь, медианный размер домохозяйства, медианная жилая площадь), используя статистические методы и обработку пространственных данных.

В данных Реформы ЖКХ в атрибутах 78.85 % домов есть население `INHAB`, там, где его нет, его можно расчитать: количество квартир `RMC_LIVE` умножается на медианный размер домохозяйств в данном городе, вычисленный по домам, где население в атрибутах есть, также если есть жилая площадь `AREA_LIVE`, то считается еще медианная площадь на человека, по данным, где есть население, далее среднее двух этих показателей, или же только количество квартир, умноженное на средний размер домохозяйств в данном городе, если данных по площади нет (квартиры есть везде).

In [16]:
pd.options.mode.chained_assignment = None

builds_lst = []
new_builds_lst = []
people_per_rmc_lst = []
area_per_person_lst = []
people_lst = []

def to_clear(value):
    if value:
        value = str(value)
        value = value.replace(',', '.')
        value = value.replace(' ', '')
        value = float(value)
        return value
    else:
        return None
    
def to_count_people(row):
    inhab = to_clear(row['INHAB'])
    rmc = to_clear(row['RMC_LIVE'])
    area = to_clear(row['AREA_LIVE'])
    area_per_person = row['area_per_person']
    people_per_rmc = row['people_per_rmc']
    if inhab:
        if inhab>0:
            return inhab
    else:
        if rmc and area:
            rmc = float(rmc)
            area = float(area)
            if rmc > 0 and area > 0:
                inhab = (area/area_per_person + rmc*people_per_rmc) / 2
            elif area > 0:
                inhab = area/area_per_person
            elif rmc > 0:
                inhab = rmc*people_per_rmc
            else:
                inhab = 0
        elif rmc:
            rmc = float(rmc)
            if rmc > 0:
                inhab = rmc*people_per_rmc
            else:
                inhab = 0
        else:
            return None
    return inhab

people_area = builds_clipped[['AREA_LIVE', 'INHAB']].dropna()
people_area['INHAB'] = people_area['INHAB'].astype('str')
people_area['INHAB'] = people_area['INHAB'].str.replace(' ', '')
people_area['INHAB'] = people_area['INHAB'].str.replace(',', '.')
people_area['INHAB'] = people_area['INHAB'].astype('float')

people_area['AREA_LIVE'] = people_area['AREA_LIVE'].astype('str')
people_area['AREA_LIVE'] = people_area['AREA_LIVE'].str.replace(' ', '')
people_area['AREA_LIVE'] = people_area['AREA_LIVE'].str.replace(',', '.')
people_area['AREA_LIVE'] = people_area['AREA_LIVE'].astype('float')
people_area = people_area[people_area['INHAB']>0]

people_rmc = builds_clipped[['RMC_LIVE', 'INHAB']].dropna()
people_rmc['INHAB'] = people_rmc['INHAB'].astype('str')
people_rmc['INHAB'] = people_rmc['INHAB'].str.replace(' ', '')
people_rmc['INHAB'] = people_rmc['INHAB'].str.replace(',', '.')
people_rmc['INHAB'] = people_rmc['INHAB'].astype('float')

people_rmc['RMC_LIVE'] = people_rmc['RMC_LIVE'].astype('str')
people_rmc['RMC_LIVE'] = people_rmc['RMC_LIVE'].str.replace(' ', '')
people_rmc['RMC_LIVE'] = people_rmc['RMC_LIVE'].str.replace(',', '.')
people_rmc['RMC_LIVE'] = people_rmc['RMC_LIVE'].astype('float')
people_rmc = people_rmc[people_rmc['RMC_LIVE']>0]

area_per_person = np.median(people_area['AREA_LIVE']/people_area['INHAB'])
people_per_rmc = np.median(people_rmc['INHAB']/people_rmc['RMC_LIVE'])

builds_clipped['area_per_person'] = area_per_person
builds_clipped['people_per_rmc'] = people_per_rmc
builds_clipped['area_per_person'] = area_per_person
builds_clipped['people_per_rmc'] = people_per_rmc
builds_clipped['INHAB_count'] = builds_clipped.apply(to_count_people, axis=1)


print(f'Amount of the buildings before and after clipping: {len(builds)} -> {len(builds_clipped)}')
print(f'Amount of the new buildings before and after clipping: {len(new_builds)} -> {len(new_builds_clipped)}')
print(f'\nМедианная жилая площадь на 1 человека: {area_per_person:.2f} м2')
print(f'Размер выборки для расчета: {len(people_area)}')

print(f'\nМедианное число домохозяйств: {people_per_rmc:.2f}')
print(f'Размер выборки для расчета: {len(people_rmc)}')

Amount of the buildings before and after clipping: 34375 -> 34375
Amount of the new buildings before and after clipping: 2067 -> 2067

Медианная жилая площадь на 1 человека: 21.51 м2
Размер выборки для расчета: 27099

Медианное число домохозяйств: 2.38
Размер выборки для расчета: 25364


### Обрабатываем некоторые атрибуты

In [17]:
new_builds_clipped['year_build'] = new_builds_clipped['data.objReady100PercDt'].apply(lambda x: int(x[:4]) if x else None)

def to_def_class_energy(value):
    if value in ['E', 'F', 'G']:
        return 2
    elif value in ['C', 'D']:
        return 1
    elif value in ['B', 'B+', 'B++', 'A', 'A+', 'A++']:
        return 0
    else:
        return None

builds_clipped['energy_class'] = builds_clipped['ENRG_CLS'].apply(to_def_class_energy)

def to_mat(value):
    value = str(value).lower()
    if value=='кирпич' or value=='железобетон' or value=='блочные' or value=='панельные' or value=='монолитные' or value=='смешанные' or value=='деревянные':
        return value
    elif value!='none':
        return 'иные'
    else:
        None
builds_clipped['material']= builds_clipped['MAT_NES'].apply(to_mat)
builds_clipped = pd.get_dummies(builds_clipped, columns=['material'],dtype=int, prefix='material')

builds_clipped['BLAG_SPORT'] = builds_clipped['BLAG_SPORT'].apply(lambda x: 1 if x=='Да' else 0)
builds_clipped['BLAG_PLAY'] = builds_clipped['BLAG_PLAY'].apply(lambda x: 1 if x=='Да' else 0)

builds_clipped['RMC_LIVE'] = builds_clipped['RMC_LIVE'].astype('str')
builds_clipped['RMC_LIVE'] = builds_clipped['RMC_LIVE'].str.replace(' ', '')
builds_clipped['RMC_LIVE'] = builds_clipped['RMC_LIVE'].str.replace(',', '.')
builds_clipped['RMC_LIVE'] = builds_clipped['RMC_LIVE'].replace('None', np.nan)
builds_clipped['RMC_LIVE'] = builds_clipped['RMC_LIVE'].astype('float')

builds_clipped['AREA_LIVE'] = builds_clipped['AREA_LIVE'].astype('str')
builds_clipped['AREA_LIVE'] = builds_clipped['AREA_LIVE'].str.replace(' ', '')
builds_clipped['AREA_LIVE'] = builds_clipped['AREA_LIVE'].str.replace(',', '.')
builds_clipped['AREA_LIVE'] = builds_clipped['AREA_LIVE'].replace('None', np.nan)
builds_clipped['AREA_LIVE'] = builds_clipped['AREA_LIVE'].astype('float')

builds_clipped[['RMC_LIVE',
'AREA_LIVE','energy_class','BLAG_SPORT','BLAG_PLAY', 'material_блочные', 'material_деревянные',
       'material_железобетон', 'material_иные', 'material_кирпич',
       'material_монолитные', 'material_панельные', 'material_смешанные']]

,RMC_LIVE,AREA_LIVE,energy_class,BLAG_SPORT,BLAG_PLAY,material_блочные,material_деревянные,material_железобетон,material_иные,material_кирпич,material_монолитные,material_панельные,material_смешанные
0,16.0,789.10,NaN,1,1,0,0,0,0,1,0,0,0
1,60.0,2968.00,NaN,0,1,0,0,0,0,0,0,0,0
2,61.0,3044.00,NaN,0,1,0,0,0,0,0,0,0,0
3,40.0,2078.00,NaN,1,1,0,0,0,0,0,1,0,0
4,60.0,3028.00,NaN,0,1,0,0,0,0,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
34370,80.0,4200.95,NaN,1,1,0,0,0,0,1,0,0,0
34371,24.0,978.05,NaN,0,1,0,0,0,0,1,0,0,0
34372,10.0,597.63,NaN,0,0,0,0,0,0,1,0,0,0
34373,12.0,635.00,NaN,0,0,0,0,0,0,1,0,0,0


In [18]:
new_builds_clipped.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
Index: 2067 entries, 0 to 1965
Data columns (total 16 columns):
 #   Column                          Non-Null Count  Dtype   
---  ------                          --------------  -----   
 0   просто                          2067 non-null   bool    
 1   data.id                         2067 non-null   float64 
 2   data.developer.devId            2067 non-null   float64 
 3   data.developer.devShortCleanNm  2067 non-null   object  
 4   data.region                     2067 non-null   object  
 5   data.address                    2067 non-null   object  
 6   data.objElemLivingCnt           2067 non-null   object  
 7   data.objReady100PercDt          2065 non-null   object  
 8   data.objLkFinishTypeCount       2067 non-null   object  
 9   data.objElemParkingCnt          2067 non-null   object  
 10  data.objSquareLiving            2067 non-null   object  
 11  data.objLkLatitude              2067 non-null   float64 
 12  data.objLkLongitu

In [19]:
builds_clipped.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
Index: 34375 entries, 0 to 34374
Data columns (total 60 columns):
 #   Column                Non-Null Count  Dtype   
---  ------                --------------  -----   
 0   LAT                   34375 non-null  float64 
 1   LON                   34375 non-null  float64 
 2   HOUSE_ID              34375 non-null  int64   
 3   ADDRESS               34375 non-null  object  
 4   FIAS                  34375 non-null  int64   
 5   YEAR_BLD              31454 non-null  float64 
 6   YEAR_EXPL             33144 non-null  float64 
 7   SERIE                 32255 non-null  object  
 8   HOUSE_TYPE            34299 non-null  object  
 9   CAPFOND               33414 non-null  object  
 10  MGMT                  32598 non-null  object  
 11  MGMT_LINK             32598 non-null  object  
 12  AVAR                  0 non-null      float64 
 13  LEVELS                33222 non-null  float64 
 14  DOORS                 32095 non-null  object  
 15 

In [20]:
builds_clipped.columns

Index(['LAT', 'LON', 'HOUSE_ID', 'ADDRESS', 'FIAS', 'YEAR_BLD', 'YEAR_EXPL',
       'SERIE', 'HOUSE_TYPE', 'CAPFOND', 'MGMT', 'MGMT_LINK', 'AVAR', 'LEVELS',
       'DOORS', 'LIFTS', 'RMC', 'RMC_LIVE', 'RMC_NLIVE', 'INHAB', 'AREA',
       'AREA_LIVE', 'AREA_NLIVE', 'AREA_GEN', 'AREA_LAND', 'AREA_PARK',
       'CADNO', 'ENRG_CLS', 'BLAG_PLAY', 'BLAG_SPORT', 'BLAG_OTHER', 'OTHER',
       'COLD_WATER', 'HOT_WATER', 'FUNDAMENT', 'PEREKRYT', 'MAT_NES',
       'INOE_OBOR', 'ELECT_TYPE', 'ELECT_VVOD', 'TEPLO_TYPE', 'GORVODA_TY',
       'HOLVODA_TY', 'VOTV_TYPE', 'VOTV_YAM', 'GAZ_TYPE', 'VSTK_TYPE',
       'area_per_person', 'people_per_rmc', 'INHAB_count', 'geometry',
       'energy_class', 'material_блочные', 'material_деревянные',
       'material_железобетон', 'material_иные', 'material_кирпич',
       'material_монолитные', 'material_панельные', 'material_смешанные'],
      dtype='object')

In [21]:
builds_clipped[['INHAB_count','YEAR_BLD','YEAR_EXPL','LEVELS','RMC_LIVE',
'AREA_LIVE','energy_class','BLAG_SPORT','BLAG_PLAY', 'material_блочные', 'material_деревянные',
       'material_железобетон', 'material_иные', 'material_кирпич',
       'material_монолитные', 'material_панельные', 'material_смешанные']].describe()

,INHAB_count,YEAR_BLD,YEAR_EXPL,LEVELS,RMC_LIVE,AREA_LIVE,energy_class,BLAG_SPORT,BLAG_PLAY,material_блочные,material_деревянные,material_железобетон,material_иные,material_кирпич,material_монолитные,material_панельные,material_смешанные
count,32775.000000,31454.000000,33144.000000,33222.000000,31034.000000,33201.000000,11636.000000,34375.000000,34375.000000,34375.000000,34375.000000,34375.000000,34375.000000,34375.000000,34375.000000,34375.000000,34375.000000
mean,289.706334,1971.410981,1971.572049,9.793571,127.424438,6521.973999,1.032056,0.205935,0.502080,0.137280,0.003753,0.201687,0.011055,0.324480,0.108975,0.113455,0.019840
std,303.121994,26.617726,26.441464,5.577216,120.127363,6936.035859,0.752195,0.404389,0.500003,0.344148,0.061145,0.401266,0.104559,0.468187,0.311612,0.317152,0.139452
min,1.000000,169.000000,169.000000,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,120.000000,1960.000000,1960.000000,5.000000,60.000000,2568.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,184.000000,1969.000000,1969.000000,9.000000,90.000000,4187.070000,1.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,370.000000,1986.000000,1986.000000,14.000000,160.000000,8438.550000,2.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000
max,5409.086091,2023.000000,2023.000000,58.000000,4455.000000,159917.000000,2.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [22]:
builds_clipped.to_csv('builds_clipped.csv', index=False)
new_builds_clipped.to_csv('new_builds_clipped.csv', index=False)

### Все многоквартирные дома
- 'YEAR_BLD' - год постройки
- 'YEAR_EXPL' - год эксплуатации
- 'MGMT' - Управляющая компания (их более 2489)
- 'LEVELS' - этажи
- 'DOORS'- подъезды
- 'MAT_NES' - матерал
- 'RMC_LIVE' - жилые помещения
- 'AREA_LIVE' - жилая площадь
- 'ENRG_CLS' - класс энергоэффективности дома
- 'energy_class' - 2 (плохая энергоэф.), 1 - норм, 0 - хорошая
- 'BLAG_SPORT' - спортивная площадка
- 'BLAG_PLAY' - детская площадка


### Новостройки
- 'data.objLkClassDesc' - класс (типовой, комфорт, бизнес, элитный, другое)
- 'data.objSquareLiving' - жилая площадь
- 'data.objElemParkingCnt' - количество парковочных мест
- 'data.objReady100PercDt' - дата постройки
- 'data.objElemLivingCnt' - количество квартир


1) Высокие показатели энергоэффективности многоквартирного дома
Это классы от A++ до B. Как правило, такие классы присваивают новостройкам из сегментов бизнес и премиум. Благодаря высокому качеству стройматериалов и коммуникаций в таких домах экономия ресурсов может достигать 60%. Если разбивать каждый класс, то получатся следующие показатели экономии:

- A+ — 50-60%;
- A — 40-50%;
- B — 30-40%.

2) Нормальные показатели энергоэффективности многоквартирного дома
Сюда относятся классы C — повышенный — и D — нормальный. Если дому присвоен С, жители многоквартирного дома могут в совокупности сэкономить до 30% ресурсов. Если D, до 15%.

3) Низкие показатели энергоэффективности многоквартирного дома
При низких классах дома «теряют» до половины ресурсов впустую:

- E — до 25% энергии;
- F — от 25% до 50%;
- G — свыше 50%.

## Пешеходные изохроны 15 мин

In [ ]:
'YEAR_BLD' - среднее, медиана,
'YEAR_EXPL' - среднее, медиана,
'LEVELS'- среднее, медиана,
'RMC_LIVE'- среднее, медиана,
'AREA_LIVE'- среднее, медиана,
'energy_class'- среднее, медиана,
'INHAB_count' - сумма,
'BLAG_SPORT' - сумма,
'BLAG_PLAY' - сумма,
'material_блочные' - сумма,
'material_деревянные' - сумма,
'material_железобетон' - сумма,
'material_иные' - сумма,
'material_кирпич' - сумма,
'material_монолитные' - сумма,
'material_панельные' - сумма,
'material_смешанные' - сумма

In [37]:
import pandas as pd
import geopandas as gpd
import osmnx as ox
import networkx as nx
from shapely.geometry import Point
import time
import numpy as np
import warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', category=RuntimeWarning, message='Mean of empty slice')

# --- НАСТРОЙКИ ---
GRAPH_PATH = r'D:/Users/Mariia/Desktop/Курсач/4 курс/2gis' # Путь к папке с графами
WALK_SPEED_KMH = 4.5
METERS_PER_MIN = WALK_SPEED_KMH * 1000 / 60
WALK_TIME_MIN = 15

# --- ЗАГРУЗКА ГРАФА ---
print("Загружаем пешеходный граф Москвы...")
G_walk = ox.load_graphml(f'{GRAPH_PATH}/moscow_walk.graphml')
G_walk = ox.project_graph(G_walk) # Проекция в метры (UTM)
current_crs = G_walk.graph['crs']

# Добавляем атрибут времени на ребра графа (если его еще нет)
for u, v, k, data in G_walk.edges(keys=True, data=True):
    length = data.get('length')
    if length is not None:
        data['time'] = float(length) / METERS_PER_MIN

# --- ПОДГОТОВКА ДОМОВ КАК GeoDataFrame ---
# Предполагается, что builds_clipped уже существует как DataFrame с LAT, LON и нужными столбцами
# Если нет геометрии — создаем её
# builds_clipped = pd.read_csv('builds_clipped.csv')
builds_clipped['geometry'] = gpd.points_from_xy(builds_clipped.LON, builds_clipped.LAT)
gdf_builds = gpd.GeoDataFrame(builds_clipped, geometry='geometry', crs="EPSG:4326")

# Проецируем дома в ту же систему координат, что и граф
gdf_builds = gdf_builds.to_crs(current_crs)

# --- НАЧАЛО ОБРАБОТКИ ---
print('НАЧАЛО ОБРАБОТКИ')
fitness = pd.read_csv('data/fitness_w_centerd.csv')
results = []
print(f"Начинаем обработку {len(fitness)} объектов...")

start = time.time()  # или time.perf_counter()

for idx, row in fitness.iterrows():
    lat, lon = row['lat'], row['lon']
    
    if idx % 250 == 0: # Выводим прогресс реже
        print(f"Обрабатываем объект {idx+1}/{len(fitness)}...")
    
    try:
        # --- ПОСТРОЕНИЕ ИЗОХРОНЫ ---
        point_wgs84 = Point(lon, lat)
        point_graph_crs = gpd.GeoSeries([point_wgs84], crs="EPSG:4326").to_crs(current_crs).iloc[0]
        
        center_node = ox.distance.nearest_nodes(G_walk, X=point_graph_crs.x, Y=point_graph_crs.y)
        subgraph = nx.ego_graph(G_walk, center_node, radius=WALK_TIME_MIN, distance='time')
        
        if len(subgraph.nodes()) < 3:
            # Если изохрона не построилась — записываем NaN для всех метрик
            empty_row = {'fitness_id': idx}
            for col in ['INHAB_count', 'YEAR_BLD_mean', 'YEAR_BLD_median', 'YEAR_EXPL_mean', 'YEAR_EXPL_median',
                        'LEVELS_mean', 'LEVELS_median', 'RMC_LIVE_mean', 'RMC_LIVE_median', 'AREA_LIVE_mean',
                        'AREA_LIVE_median', 'energy_class_mean', 'energy_class_median', 'BLAG_SPORT_sum',
                        'BLAG_PLAY_sum'] + [f'material_{m}_sum' for m in [
                            'блочные','деревянные','железобетон','иные','кирпич','монолитные','панельные','смешанные']]:
                empty_row[col] = np.nan
            results.append(empty_row)
            continue

        poly = gpd.GeoSeries([Point(data['x'], data['y']) for _, data in subgraph.nodes(data=True)]).unary_union.convex_hull

        # --- ПРОСТРАНСТВЕННОЕ СОЕДИНЕНИЕ ---
        hits = gpd.sjoin(gpd.GeoDataFrame(geometry=[poly], crs=current_crs), gdf_builds, how='inner', predicate='intersects')
        
        # Если ничего не найдено — записываем NaN для всех метрик
        if hits.empty:
            empty_row = {'fitness_id': idx}
            for col in ['INHAB_count', 'YEAR_BLD_mean', 'YEAR_BLD_median', 'YEAR_EXPL_mean', 'YEAR_EXPL_median',
                        'LEVELS_mean', 'LEVELS_median', 'RMC_LIVE_mean', 'RMC_LIVE_median', 'AREA_LIVE_mean',
                        'AREA_LIVE_median', 'energy_class_mean', 'energy_class_median', 'BLAG_SPORT_sum',
                        'BLAG_PLAY_sum'] + [f'material_{m}_sum' for m in [
                            'блочные','деревянные','железобетон','иные','кирпич','монолитные','панельные','смешанные']]:
                empty_row[col] = np.nan
            results.append(empty_row)
            continue

        # --- РАСЧЕТ МЕТРИК ---
        row_result = {'fitness_id': idx}
        
        # 1. Сумма INHAB_count (игнорируем NaN)
        row_result['INHAB_count'] = hits['INHAB_count'].sum()
        
        # 2. Средние и медианы для числовых столбцов (игнорируем NaN)
        for col_base in ['YEAR_BLD', 'YEAR_EXPL', 'LEVELS', 'RMC_LIVE', 'AREA_LIVE', 'energy_class']:
            row_result[f'{col_base}_mean'] = hits[col_base].mean()
            row_result[f'{col_base}_median'] = hits[col_base].median()
        
        # 3. Суммы для BLAG и материалов (игнорируем NaN)
        row_result['BLAG_SPORT_sum'] = hits['BLAG_SPORT'].sum()
        row_result['BLAG_PLAY_sum'] = hits['BLAG_PLAY'].sum()
        
        materials_list = ['блочные','деревянные','железобетон','иные','кирпич','монолитные','панельные','смешанные']
        for mat in materials_list:
            col_name = f'material_{mat}'
            row_result[f'{col_name}_sum'] = hits[col_name].sum()
        
        results.append(row_result)
        
    except Exception as e:
        print(f"Ошибка на индексе {idx}: {e}")
        # В случае ошибки — записываем NaN для всех метрик
        empty_row = {'fitness_id': idx}
        for col in ['INHAB_count', 'YEAR_BLD_mean', 'YEAR_BLD_median', 'YEAR_EXPL_mean', 'YEAR_EXPL_median',
                    'LEVELS_mean', 'LEVELS_median', 'RMC_LIVE_mean', 'RMC_LIVE_median', 'AREA_LIVE_mean',
                    'AREA_LIVE_median', 'energy_class_mean', 'energy_class_median', 'BLAG_SPORT_sum',
                    'BLAG_PLAY_sum'] + [f'material_{m}_sum' for m in [
                        'блочные','деревянные','железобетон','иные','кирпич','монолитные','панельные','смешанные']]:
            empty_row[col] = np.nan
        results.append(empty_row)

# --- СОХРАНЕНИЕ РЕЗУЛЬТАТА ---
res_df = pd.DataFrame(results)
res_df.head()

Загружаем пешеходный граф Москвы...
НАЧАЛО ОБРАБОТКИ
Начинаем обработку 5029 объектов...
Обрабатываем объект 1/5029...
Обрабатываем объект 251/5029...
Обрабатываем объект 501/5029...
Обрабатываем объект 751/5029...
Обрабатываем объект 1001/5029...
Обрабатываем объект 1251/5029...
Обрабатываем объект 1501/5029...
Обрабатываем объект 1751/5029...
Обрабатываем объект 2001/5029...
Обрабатываем объект 2251/5029...
Обрабатываем объект 2501/5029...
Обрабатываем объект 2751/5029...
Обрабатываем объект 3001/5029...
Обрабатываем объект 3251/5029...
Обрабатываем объект 3501/5029...
Обрабатываем объект 3751/5029...
Обрабатываем объект 4001/5029...
Обрабатываем объект 4251/5029...
Обрабатываем объект 4501/5029...
Обрабатываем объект 4751/5029...
Обрабатываем объект 5001/5029...


,fitness_id,INHAB_count,YEAR_BLD_mean,YEAR_BLD_median,YEAR_EXPL_mean,YEAR_EXPL_median,LEVELS_mean,LEVELS_median,RMC_LIVE_mean,RMC_LIVE_median,...,BLAG_SPORT_sum,BLAG_PLAY_sum,material_блочные_sum,material_деревянные_sum,material_железобетон_sum,material_иные_sum,material_кирпич_sum,material_монолитные_sum,material_панельные_sum,material_смешанные_sum
0,0,23542.198003,1917.917404,1917.0,1918.088825,1916.0,5.240688,5.0,28.831832,19.0,...,10.0,59.0,2.0,0.0,0.0,4.0,249.0,81.0,1.0,4.0
1,1,17905.000000,1955.414141,1959.0,1956.198020,1960.0,8.138614,8.0,83.255102,78.0,...,11.0,54.0,15.0,0.0,1.0,0.0,78.0,0.0,6.0,0.0
2,2,1533.000000,1968.250000,1962.0,1968.250000,1962.0,7.750000,8.0,98.125000,112.0,...,1.0,1.0,0.0,0.0,0.0,0.0,6.0,1.0,0.0,0.0
3,3,2021.000000,2016.142857,2017.0,2015.600000,2015.0,7.033333,4.0,224.833333,170.0,...,8.0,17.0,0.0,0.0,0.0,2.0,1.0,5.0,7.0,4.0
4,4,4161.877857,1933.200000,1938.0,1933.200000,1938.0,6.160000,5.0,63.280000,52.0,...,15.0,22.0,1.0,0.0,0.0,0.0,22.0,0.0,0.0,1.0


In [38]:
res_df = pd.DataFrame(results)
print("Готово!")
end = time.time()
print(f"Время: {end - start:.4f} сек")

lst_material = []
for col in list(res_df.columns):
    if 'material' in col:
        lst_material.append(col)
res_df['materials'] = res_df[lst_material].sum(axis=1)

for col in list(res_df.columns):
    if 'material' in col:
        res_df[col] = res_df[col]/res_df['materials']

for col in list(res_df.columns[1:]):
    old_name = col
    new_name = col + '_15ped'
    res_df = res_df.rename(columns={old_name: new_name})
    
fitness_iso_pop = pd.concat([fitness,res_df[1:-1]], axis=1)
fitness_iso_pop.to_csv('fitness_pop_iso_15ped.csv', index = False)
fitness_iso_pop.to_excel('fitness_pop_iso_15ped.xlsx', index = False)

Готово!
Время: 19197.4437 сек


## Автомобильные изохроны 15 мин

In [39]:
import pandas as pd
import geopandas as gpd
import osmnx as ox
import networkx as nx
from shapely.geometry import Point
import time
import numpy as np
import warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', category=RuntimeWarning, message='Mean of empty slice')

# --- НАСТРОЙКИ ---
GRAPH_PATH = r'D:/Users/Mariia/Desktop/Курсач/4 курс/2gis' # Путь к папке с графами

# --- ЗАГРУЗКА ГРАФА ---
print("Загружаем пешеходный граф Москвы...")
G_drive = ox.load_graphml(f'{GRAPH_PATH}/moscow_drive.graphml')
G_drive = ox.project_graph(G_drive) # Проекция в метры (UTM)
current_crs = G_drive.graph['crs']

DRIVE_SPEED_KMH = 30 # Средняя скорость авто в городе
METERS_PER_MIN_DRIVE = DRIVE_SPEED_KMH * 1000 / 60
DRIVE_TIME_MIN = 15
MAX_DRIVE_METERS = DRIVE_TIME_MIN * METERS_PER_MIN_DRIVE

# Добавляем атрибут времени на ребра графа (если его еще нет)
for u, v, k, data in G_drive.edges(keys=True, data=True):
    length = data.get('length')
    if length is not None:
        data['time'] = float(length) / METERS_PER_MIN_DRIVE

# --- ПОДГОТОВКА ДОМОВ КАК GeoDataFrame ---
# Предполагается, что builds_clipped уже существует как DataFrame с LAT, LON и нужными столбцами
# Если нет геометрии — создаем её
# builds_clipped = pd.read_csv('builds_clipped.csv')
builds_clipped['geometry'] = gpd.points_from_xy(builds_clipped.LON, builds_clipped.LAT)
gdf_builds = gpd.GeoDataFrame(builds_clipped, geometry='geometry', crs="EPSG:4326")

# Проецируем дома в ту же систему координат, что и граф
gdf_builds = gdf_builds.to_crs(current_crs)

# --- НАЧАЛО ОБРАБОТКИ ---
print('НАЧАЛО ОБРАБОТКИ')
fitness = pd.read_csv('data/fitness_w_centerd.csv')
results = []
print(f"Начинаем обработку {len(fitness)} объектов...")

start = time.time()  # или time.perf_counter()

for idx, row in fitness.iterrows():
    lat, lon = row['lat'], row['lon']
    
    if idx % 250 == 0: # Выводим прогресс реже
        print(f"Обрабатываем объект {idx+1}/{len(fitness)}...")
    
    try:
        # --- ПОСТРОЕНИЕ ИЗОХРОНЫ ---
        point_wgs84 = Point(lon, lat)
        point_graph_crs = gpd.GeoSeries([point_wgs84], crs="EPSG:4326").to_crs(current_crs).iloc[0]
        
        center_node = ox.distance.nearest_nodes(G_drive, X=point_graph_crs.x, Y=point_graph_crs.y)
        subgraph = nx.ego_graph(G_drive, center_node, radius=DRIVE_TIME_MIN, distance='time')
        
        if len(subgraph.nodes()) < 3:
            # Если изохрона не построилась — записываем NaN для всех метрик
            empty_row = {'fitness_id': idx}
            for col in ['INHAB_count', 'YEAR_BLD_mean', 'YEAR_BLD_median', 'YEAR_EXPL_mean', 'YEAR_EXPL_median',
                        'LEVELS_mean', 'LEVELS_median', 'RMC_LIVE_mean', 'RMC_LIVE_median', 'AREA_LIVE_mean',
                        'AREA_LIVE_median', 'energy_class_mean', 'energy_class_median', 'BLAG_SPORT_sum',
                        'BLAG_PLAY_sum'] + [f'material_{m}_sum' for m in [
                            'блочные','деревянные','железобетон','иные','кирпич','монолитные','панельные','смешанные']]:
                empty_row[col] = np.nan
            results.append(empty_row)
            continue

        poly = gpd.GeoSeries([Point(data['x'], data['y']) for _, data in subgraph.nodes(data=True)]).unary_union.convex_hull

        # --- ПРОСТРАНСТВЕННОЕ СОЕДИНЕНИЕ ---
        hits = gpd.sjoin(gpd.GeoDataFrame(geometry=[poly], crs=current_crs), gdf_builds, how='inner', predicate='intersects')
        
        # Если ничего не найдено — записываем NaN для всех метрик
        if hits.empty:
            empty_row = {'fitness_id': idx}
            for col in ['INHAB_count', 'YEAR_BLD_mean', 'YEAR_BLD_median', 'YEAR_EXPL_mean', 'YEAR_EXPL_median',
                        'LEVELS_mean', 'LEVELS_median', 'RMC_LIVE_mean', 'RMC_LIVE_median', 'AREA_LIVE_mean',
                        'AREA_LIVE_median', 'energy_class_mean', 'energy_class_median', 'BLAG_SPORT_sum',
                        'BLAG_PLAY_sum'] + [f'material_{m}_sum' for m in [
                            'блочные','деревянные','железобетон','иные','кирпич','монолитные','панельные','смешанные']]:
                empty_row[col] = np.nan
            results.append(empty_row)
            continue

        # --- РАСЧЕТ МЕТРИК ---
        row_result = {'fitness_id': idx}
        
        # 1. Сумма INHAB_count (игнорируем NaN)
        row_result['INHAB_count'] = hits['INHAB_count'].sum()
        
        # 2. Средние и медианы для числовых столбцов (игнорируем NaN)
        for col_base in ['YEAR_BLD', 'YEAR_EXPL', 'LEVELS', 'RMC_LIVE', 'AREA_LIVE', 'energy_class']:
            row_result[f'{col_base}_mean'] = hits[col_base].mean()
            row_result[f'{col_base}_median'] = hits[col_base].median()
        
        # 3. Суммы для BLAG и материалов (игнорируем NaN)
        row_result['BLAG_SPORT_sum'] = hits['BLAG_SPORT'].sum()
        row_result['BLAG_PLAY_sum'] = hits['BLAG_PLAY'].sum()
        
        materials_list = ['блочные','деревянные','железобетон','иные','кирпич','монолитные','панельные','смешанные']
        for mat in materials_list:
            col_name = f'material_{mat}'
            row_result[f'{col_name}_sum'] = hits[col_name].sum()
        
        results.append(row_result)
        
    except Exception as e:
        print(f"Ошибка на индексе {idx}: {e}")
        # В случае ошибки — записываем NaN для всех метрик
        empty_row = {'fitness_id': idx}
        for col in ['INHAB_count', 'YEAR_BLD_mean', 'YEAR_BLD_median', 'YEAR_EXPL_mean', 'YEAR_EXPL_median',
                    'LEVELS_mean', 'LEVELS_median', 'RMC_LIVE_mean', 'RMC_LIVE_median', 'AREA_LIVE_mean',
                    'AREA_LIVE_median', 'energy_class_mean', 'energy_class_median', 'BLAG_SPORT_sum',
                    'BLAG_PLAY_sum'] + [f'material_{m}_sum' for m in [
                        'блочные','деревянные','железобетон','иные','кирпич','монолитные','панельные','смешанные']]:
            empty_row[col] = np.nan
        results.append(empty_row)

# --- СОХРАНЕНИЕ РЕЗУЛЬТАТА ---
res_df2 = pd.DataFrame(results)
res_df2.head()

Загружаем пешеходный граф Москвы...
НАЧАЛО ОБРАБОТКИ
Начинаем обработку 5029 объектов...
Обрабатываем объект 1/5029...
Обрабатываем объект 251/5029...
Обрабатываем объект 501/5029...
Обрабатываем объект 751/5029...
Обрабатываем объект 1001/5029...
Обрабатываем объект 1251/5029...
Обрабатываем объект 1501/5029...
Обрабатываем объект 1751/5029...
Обрабатываем объект 2001/5029...
Обрабатываем объект 2251/5029...
Обрабатываем объект 2501/5029...
Обрабатываем объект 2751/5029...
Обрабатываем объект 3001/5029...
Обрабатываем объект 3251/5029...
Обрабатываем объект 3501/5029...
Обрабатываем объект 3751/5029...
Обрабатываем объект 4001/5029...
Обрабатываем объект 4251/5029...
Обрабатываем объект 4501/5029...
Обрабатываем объект 4751/5029...
Обрабатываем объект 5001/5029...


,fitness_id,INHAB_count,YEAR_BLD_mean,YEAR_BLD_median,YEAR_EXPL_mean,YEAR_EXPL_median,LEVELS_mean,LEVELS_median,RMC_LIVE_mean,RMC_LIVE_median,...,BLAG_SPORT_sum,BLAG_PLAY_sum,material_блочные_sum,material_деревянные_sum,material_железобетон_sum,material_иные_sum,material_кирпич_sum,material_монолитные_sum,material_панельные_sum,material_смешанные_sum
0,0,957436.027967,1950.289370,1959.0,1950.162925,1959.0,7.746906,6.0,73.685162,60.0,...,1037.0,3235.0,680.0,20.0,263.0,58.0,3949.0,659.0,184.0,84.0
1,1,901886.734714,1953.061737,1959.0,1953.139088,1959.0,7.836503,6.0,76.203371,60.0,...,831.0,2635.0,800.0,10.0,229.0,75.0,3520.0,471.0,254.0,187.0
2,2,624876.965453,1952.362664,1958.0,1952.163431,1958.0,7.895757,6.0,81.208394,60.0,...,483.0,1722.0,309.0,13.0,222.0,35.0,2582.0,373.0,115.0,56.0
3,3,203347.379132,1999.838284,2001.0,1998.252149,2001.0,10.237822,10.0,160.575261,125.0,...,172.0,349.0,16.0,0.0,264.0,21.0,85.0,92.0,114.0,30.0
4,4,770899.279271,1949.209732,1958.0,1948.935770,1958.0,7.589622,6.0,76.083487,59.0,...,1008.0,2378.0,469.0,12.0,267.0,30.0,3070.0,459.0,134.0,70.0


In [41]:
res_df2

,fitness_id,INHAB_count,YEAR_BLD_mean,YEAR_BLD_median,YEAR_EXPL_mean,YEAR_EXPL_median,LEVELS_mean,LEVELS_median,RMC_LIVE_mean,RMC_LIVE_median,...,BLAG_PLAY_sum,material_блочные_sum,material_деревянные_sum,material_железобетон_sum,material_иные_sum,material_кирпич_sum,material_монолитные_sum,material_панельные_sum,material_смешанные_sum,materials
0,0,957436.027967,1950.289370,1959.0,1950.162925,1959.0,7.746906,6.0,73.685162,60.0,...,3235.0,680.0,20.0,263.0,58.0,3949.0,659.0,184.0,84.0,5897.0
1,1,901886.734714,1953.061737,1959.0,1953.139088,1959.0,7.836503,6.0,76.203371,60.0,...,2635.0,800.0,10.0,229.0,75.0,3520.0,471.0,254.0,187.0,5546.0
2,2,624876.965453,1952.362664,1958.0,1952.163431,1958.0,7.895757,6.0,81.208394,60.0,...,1722.0,309.0,13.0,222.0,35.0,2582.0,373.0,115.0,56.0,3705.0
3,3,203347.379132,1999.838284,2001.0,1998.252149,2001.0,10.237822,10.0,160.575261,125.0,...,349.0,16.0,0.0,264.0,21.0,85.0,92.0,114.0,30.0,622.0
4,4,770899.279271,1949.209732,1958.0,1948.935770,1958.0,7.589622,6.0,76.083487,59.0,...,2378.0,469.0,12.0,267.0,30.0,3070.0,459.0,134.0,70.0,4511.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5024,5024,408923.710835,1984.738636,1983.0,1985.110664,1984.0,13.088442,14.0,191.626208,143.0,...,549.0,44.0,11.0,409.0,12.0,135.0,132.0,221.0,14.0,978.0
5025,5025,308815.077110,1987.462629,1988.0,1987.838180,1988.0,12.523990,14.0,192.597870,151.0,...,485.0,16.0,13.0,302.0,12.0,126.0,107.0,185.0,13.0,774.0
5026,5026,569734.803293,1976.488449,1976.0,1977.022883,1976.0,12.355133,12.0,190.564597,143.0,...,740.0,211.0,0.0,321.0,17.0,46.0,219.0,465.0,2.0,1281.0
5027,5027,532775.433293,1976.641373,1976.0,1977.274167,1976.0,12.180233,12.0,194.020295,143.0,...,711.0,164.0,0.0,238.0,17.0,53.0,186.0,513.0,4.0,1175.0


In [ ]:
Готово!
Время: 1910.8586 сек

In [42]:
# res_df2 = pd.DataFrame(results)
# print("Готово!")
# end = time.time()
# print(f"Время: {end - start:.4f} сек")

# lst_material = []
# for col in list(res_df2.columns):
#     if 'material' in col:
#         lst_material.append(col)
# res_df2['materials'] = res_df2[lst_material].sum(axis=1)

for col in list(res_df2.columns):
    if 'material' in col:
        res_df2[col] = res_df2[col]/res_df2['materials']

for col in list(res_df2.columns[1:]):
    old_name = col
    new_name = col + '_15drive'
    res_df2 = res_df2.rename(columns={old_name: new_name})
    
fitness_iso15drive = pd.concat([fitness, res_df2[1:-1]], axis=1)
fitness_iso15drive.to_csv('fitness_pop_iso_15drive.csv', index = False)
fitness_iso15drive.to_excel('fitness_pop_iso_15drive.xlsx', index = False)

fitness_iso15pop_15drive = pd.concat([fitness_iso_pop, res_df2[1:-1]], axis=1)
fitness_iso15pop_15drive.to_csv('fitness_pop_iso_15pop_15drive.csv', index = False)
fitness_iso15pop_15drive.to_excel('fitness_pop_iso_15pop_15drive.xlsx', index = False)

## НОВЫЕ ДОМА

- 'data.objLkClassDesc' - класс (типовой, комфорт, бизнес, элитный, другое)
- 'data.objSquareLiving' - жилая площадь
- 'data.objElemLivingCnt' - количество квартир
- 'year_build' - год постройки

- data.objLkLongitude
- data.objLkLatitude

In [66]:
new_builds_clipped_mod = new_builds_clipped.copy()
new_builds_clipped_mod = pd.get_dummies(new_builds_clipped_mod, columns=['data.objLkClassDesc'],dtype=int, prefix='newbui_class')
new_builds_clipped_mod = new_builds_clipped_mod.rename(columns={'data.objSquareLiving': 'newbui_LIVE_AREA', 'data.objElemLivingCnt': 'newbui_LIVE_RMC',
                               'year_build': 'newbui_year_build', 'data.objLkLongitude':'LON','data.objLkLatitude': 'LAT'})

new_builds_clipped_mod  = new_builds_clipped_mod [['newbui_LIVE_RMC', 'newbui_LIVE_AREA', 'LAT', 'LON', 'newbui_year_build',
       'newbui_class_Бизнес', 'newbui_class_Комфорт',
       'newbui_class_Типовой', 'newbui_class_Элитный']]
new_builds_clipped_mod['newbui_LIVE_RMC'] = new_builds_clipped_mod['newbui_LIVE_RMC'].astype('str')
new_builds_clipped_mod['newbui_LIVE_RMC'] = new_builds_clipped_mod['newbui_LIVE_RMC'].str.replace(' ', '')
new_builds_clipped_mod['newbui_LIVE_RMC'] = new_builds_clipped_mod['newbui_LIVE_RMC'].str.replace(',', '.')
new_builds_clipped_mod['newbui_LIVE_RMC'] = new_builds_clipped_mod['newbui_LIVE_RMC'].replace('None', np.nan)
new_builds_clipped_mod['newbui_LIVE_RMC'] = new_builds_clipped_mod['newbui_LIVE_RMC'].astype('float')

new_builds_clipped_mod['newbui_LIVE_AREA'] = new_builds_clipped_mod['newbui_LIVE_AREA'].astype('str')
new_builds_clipped_mod['newbui_LIVE_AREA'] = new_builds_clipped_mod['newbui_LIVE_AREA'].str.replace(' ', '')
new_builds_clipped_mod['newbui_LIVE_AREA'] = new_builds_clipped_mod['newbui_LIVE_AREA'].str.replace(',', '.')
new_builds_clipped_mod['newbui_LIVE_AREA'] = new_builds_clipped_mod['newbui_LIVE_AREA'].replace('None', np.nan)
new_builds_clipped_mod['newbui_LIVE_AREA'] = new_builds_clipped_mod['newbui_LIVE_AREA'].astype('float')

In [67]:
new_builds_clipped_mod.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2067 entries, 0 to 1965
Data columns (total 9 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   newbui_LIVE_RMC       2067 non-null   float64
 1   newbui_LIVE_AREA      2067 non-null   float64
 2   LAT                   2067 non-null   float64
 3   LON                   2067 non-null   float64
 4   newbui_year_build     2065 non-null   float64
 5   newbui_class_Бизнес   2067 non-null   int64  
 6   newbui_class_Комфорт  2067 non-null   int64  
 7   newbui_class_Типовой  2067 non-null   int64  
 8   newbui_class_Элитный  2067 non-null   int64  
dtypes: float64(5), int64(4)
memory usage: 226.0 KB


## Пешеходные изохроны 15 мин

In [46]:
print(len(new_builds_clipped))

2067


In [72]:
import pandas as pd
import geopandas as gpd
import osmnx as ox
import networkx as nx
from shapely.geometry import Point
import time
import numpy as np
import warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', category=RuntimeWarning, message='Mean of empty slice')

# --- НАСТРОЙКИ ---
GRAPH_PATH = r'D:/Users/Mariia/Desktop/Курсач/4 курс/2gis' # Путь к папке с графами
WALK_SPEED_KMH = 4.5
METERS_PER_MIN = WALK_SPEED_KMH * 1000 / 60
WALK_TIME_MIN = 15

# --- ЗАГРУЗКА ГРАФА ---
print("Загружаем пешеходный граф Москвы...")
# G_walk = ox.load_graphml(f'{GRAPH_PATH}/moscow_walk.graphml')
# G_walk = ox.project_graph(G_walk) # Проекция в метры (UTM)
current_crs = G_walk.graph['crs']

# Добавляем атрибут времени на ребра графа (если его еще нет)
for u, v, k, data in G_walk.edges(keys=True, data=True):
    length = data.get('length')
    if length is not None:
        data['time'] = float(length) / METERS_PER_MIN

# --- ПОДГОТОВКА ДОМОВ КАК GeoDataFrame ---
# Предполагается, что new_builds_clipped_mod уже существует как DataFrame с LAT, LON и нужными столбцами
# Если нет геометрии — создаем её
# new_builds_clipped_mod = pd.read_csv('new_builds_clipped_mod.csv')
new_builds_clipped_mod['geometry'] = gpd.points_from_xy(new_builds_clipped_mod.LON, new_builds_clipped_mod.LAT)
gdf_builds = gpd.GeoDataFrame(new_builds_clipped_mod, geometry='geometry', crs="EPSG:4326")

# Проецируем дома в ту же систему координат, что и граф
gdf_builds = gdf_builds.to_crs(current_crs)

# --- НАЧАЛО ОБРАБОТКИ ---
print('НАЧАЛО ОБРАБОТКИ')
fitness = pd.read_csv('data/fitness_w_centerd.csv')
results = []
print(f"Начинаем обработку {len(fitness)} объектов...")

start = time.time()  # или time.perf_counter()

for idx, row in fitness.iterrows():
    lat, lon = row['lat'], row['lon']
    
    if idx % 250 == 0: # Выводим прогресс реже
        print(f"Обрабатываем объект {idx+1}/{len(fitness)}...")
    
    try:
        # --- ПОСТРОЕНИЕ ИЗОХРОНЫ ---
        point_wgs84 = Point(lon, lat)
        point_graph_crs = gpd.GeoSeries([point_wgs84], crs="EPSG:4326").to_crs(current_crs).iloc[0]
        
        center_node = ox.distance.nearest_nodes(G_walk, X=point_graph_crs.x, Y=point_graph_crs.y)
        subgraph = nx.ego_graph(G_walk, center_node, radius=WALK_TIME_MIN, distance='time')
        
        if len(subgraph.nodes()) < 3:
            # Если изохрона не построилась — записываем NaN для всех метрик
            empty_row = {'fitness_id': idx}
            for col in ['newbui_count', 'newbui_LIVE_RMC_sum', 'newbui_LIVE_AREA_sum','newbui_year_build_mean',
                        'newbui_class_Бизнес_sum', 'newbui_class_Комфорт_sum',
                        'newbui_class_Типовой_sum', 'newbui_class_Элитный_sum']:
                empty_row[col] = np.nan
            results.append(empty_row)
            continue

        poly = gpd.GeoSeries([Point(data['x'], data['y']) for _, data in subgraph.nodes(data=True)]).unary_union.convex_hull

        # --- ПРОСТРАНСТВЕННОЕ СОЕДИНЕНИЕ ---
        hits = gpd.sjoin(gpd.GeoDataFrame(geometry=[poly], crs=current_crs), gdf_builds, how='inner', predicate='intersects')
        
        # Если ничего не найдено — записываем NaN для всех метрик
        if hits.empty:
            empty_row = {'fitness_id': idx}
            for col in ['newbui_count', 'newbui_LIVE_RMC_sum', 'newbui_LIVE_AREA_sum','newbui_year_build_mean',
                        'newbui_class_Бизнес_sum', 'newbui_class_Комфорт_sum',
                        'newbui_class_Типовой_sum', 'newbui_class_Элитный_sum']:
                empty_row[col] = np.nan
            results.append(empty_row)
            continue

        # --- РАСЧЕТ МЕТРИК ---
        row_result = {'fitness_id': idx}
        row_result['newbui_count'] = hits['LON'].count()
        row_result['newbui_LIVE_RMC_sum'] = hits['newbui_LIVE_RMC'].sum()
        row_result['newbui_LIVE_AREA_sum'] = hits['newbui_LIVE_RMC'].sum()
        row_result['newbui_class_Бизнес_sum'] = hits['newbui_LIVE_RMC'].sum()
        row_result['newbui_class_Комфорт_sum'] = hits['newbui_LIVE_RMC'].sum()
        row_result['newbui_class_Типовой_sum'] = hits['newbui_LIVE_RMC'].sum()
        row_result['newbui_class_Элитный_sum'] = hits['newbui_LIVE_RMC'].sum()
        row_result['newbui_year_build_mean'] = hits['newbui_year_build'].mean()
        
        results.append(row_result)
        
    except Exception as e:
        print(f"Ошибка на индексе {idx}: {e}")
        # В случае ошибки — записываем NaN для всех метрик
        empty_row = {'fitness_id': idx}
        for col in ['newbui_count', 'newbui_LIVE_RMC_sum', 'newbui_LIVE_AREA_sum','newbui_year_build_mean',
                        'newbui_class_Бизнес_sum', 'newbui_class_Комфорт_sum',
                        'newbui_class_Типовой_sum', 'newbui_class_Элитный_sum']:
            empty_row[col] = np.nan
        results.append(empty_row)

# --- СОХРАНЕНИЕ РЕЗУЛЬТАТА ---
res_df = pd.DataFrame(results)
res_df.head()

Загружаем пешеходный граф Москвы...
НАЧАЛО ОБРАБОТКИ
Начинаем обработку 5029 объектов...
Обрабатываем объект 1/5029...
Обрабатываем объект 251/5029...
Обрабатываем объект 501/5029...
Обрабатываем объект 751/5029...
Обрабатываем объект 1001/5029...
Обрабатываем объект 1251/5029...
Обрабатываем объект 1501/5029...
Обрабатываем объект 1751/5029...
Обрабатываем объект 2001/5029...
Обрабатываем объект 2251/5029...
Обрабатываем объект 2501/5029...
Обрабатываем объект 2751/5029...
Обрабатываем объект 3001/5029...
Обрабатываем объект 3251/5029...
Обрабатываем объект 3501/5029...
Обрабатываем объект 3751/5029...
Обрабатываем объект 4001/5029...
Обрабатываем объект 4251/5029...
Обрабатываем объект 4501/5029...
Обрабатываем объект 4751/5029...
Обрабатываем объект 5001/5029...


,fitness_id,newbui_count,newbui_LIVE_RMC_sum,newbui_LIVE_AREA_sum,newbui_class_Бизнес_sum,newbui_class_Комфорт_sum,newbui_class_Типовой_sum,newbui_class_Элитный_sum,newbui_year_build_mean
0,0,2.0,60.0,60.0,60.0,60.0,60.0,60.0,2024.000000
1,1,3.0,1793.0,1793.0,1793.0,1793.0,1793.0,1793.0,2021.333333
2,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3,39.0,15594.0,15594.0,15594.0,15594.0,15594.0,15594.0,2021.256410
4,4,10.0,4355.0,4355.0,4355.0,4355.0,4355.0,4355.0,2024.900000


In [73]:
res_df = pd.DataFrame(results)
print("Готово!")
end = time.time()
print(f"Время: {end - start:.4f} сек")

lst_material = []
for col in list(res_df.columns):
    if 'class' in col:
        lst_material.append(col)
res_df['classes'] = res_df[lst_material].sum(axis=1)

for col in list(res_df.columns):
    if 'class' in col:
        res_df[col] = res_df[col]/res_df['classes']

for col in list(res_df.columns[1:]):
    old_name = col
    new_name = col + '_15ped'
    res_df = res_df.rename(columns={old_name: new_name})
    
fitness_iso_pop = pd.concat([fitness,res_df[1:-1]], axis=1)
fitness_iso_pop.to_csv('fitness_newbui_iso_15ped.csv', index = False)
fitness_iso_pop.to_excel('fitness_newbui_iso_15ped.xlsx', index = False)

Готово!
Время: 19467.6039 сек


## Автомобильные изохроны 15 мин

In [74]:
import pandas as pd
import geopandas as gpd
import osmnx as ox
import networkx as nx
from shapely.geometry import Point
import time
import numpy as np
import warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', category=RuntimeWarning, message='Mean of empty slice')

# --- НАСТРОЙКИ ---
GRAPH_PATH = r'D:/Users/Mariia/Desktop/Курсач/4 курс/2gis' # Путь к папке с графами

DRIVE_SPEED_KMH = 30 # Средняя скорость авто в городе
METERS_PER_MIN_DRIVE = DRIVE_SPEED_KMH * 1000 / 60
DRIVE_TIME_MIN = 15
MAX_DRIVE_METERS = DRIVE_TIME_MIN * METERS_PER_MIN_DRIVE
# --- ЗАГРУЗКА ГРАФА ---
print("Загружаем пешеходный граф Москвы...")
G_drive = ox.load_graphml(f'{GRAPH_PATH}/moscow_drive.graphml')
G_drive = ox.project_graph(G_drive) # Проекция в метры (UTM)
current_crs = G_drive.graph['crs']

# Добавляем атрибут времени на ребра графа (если его еще нет)
for u, v, k, data in G_drive.edges(keys=True, data=True):
    length = data.get('length')
    if length is not None:
        data['time'] = float(length) / METERS_PER_MIN_DRIVE

# --- ПОДГОТОВКА ДОМОВ КАК GeoDataFrame ---
# Предполагается, что new_builds_clipped_mod уже существует как DataFrame с LAT, LON и нужными столбцами
# Если нет геометрии — создаем её
# new_builds_clipped_mod = pd.read_csv('new_builds_clipped_mod.csv')
new_builds_clipped_mod['geometry'] = gpd.points_from_xy(new_builds_clipped_mod.LON, new_builds_clipped_mod.LAT)
gdf_builds = gpd.GeoDataFrame(new_builds_clipped_mod, geometry='geometry', crs="EPSG:4326")

# Проецируем дома в ту же систему координат, что и граф
gdf_builds = gdf_builds.to_crs(current_crs)

# --- НАЧАЛО ОБРАБОТКИ ---
print('НАЧАЛО ОБРАБОТКИ')
fitness = pd.read_csv('data/fitness_w_centerd.csv')
results = []
print(f"Начинаем обработку {len(fitness)} объектов...")

start = time.time()  # или time.perf_counter()

for idx, row in fitness.iterrows():
    lat, lon = row['lat'], row['lon']
    
    if idx % 250 == 0: # Выводим прогресс реже
        print(f"Обрабатываем объект {idx+1}/{len(fitness)}...")
    
    try:
        # --- ПОСТРОЕНИЕ ИЗОХРОНЫ ---
        point_wgs84 = Point(lon, lat)
        point_graph_crs = gpd.GeoSeries([point_wgs84], crs="EPSG:4326").to_crs(current_crs).iloc[0]
        
        center_node = ox.distance.nearest_nodes(G_drive, X=point_graph_crs.x, Y=point_graph_crs.y)
        subgraph = nx.ego_graph(G_drive, center_node, radius=DRIVE_TIME_MIN, distance='time')
        
        if len(subgraph.nodes()) < 3:
            # Если изохрона не построилась — записываем NaN для всех метрик
            empty_row = {'fitness_id': idx}
            for col in ['newbui_count', 'newbui_LIVE_RMC_sum', 'newbui_LIVE_AREA_sum','newbui_year_build_mean',
                        'newbui_class_Бизнес_sum', 'newbui_class_Комфорт_sum',
                        'newbui_class_Типовой_sum', 'newbui_class_Элитный_sum']:
                empty_row[col] = np.nan
            results.append(empty_row)
            continue

        poly = gpd.GeoSeries([Point(data['x'], data['y']) for _, data in subgraph.nodes(data=True)]).unary_union.convex_hull

        # --- ПРОСТРАНСТВЕННОЕ СОЕДИНЕНИЕ ---
        hits = gpd.sjoin(gpd.GeoDataFrame(geometry=[poly], crs=current_crs), gdf_builds, how='inner', predicate='intersects')
        
        # Если ничего не найдено — записываем NaN для всех метрик
        if hits.empty:
            empty_row = {'fitness_id': idx}
            for col in ['newbui_count', 'newbui_LIVE_RMC_sum', 'newbui_LIVE_AREA_sum','newbui_year_build_mean',
                        'newbui_class_Бизнес_sum', 'newbui_class_Комфорт_sum',
                        'newbui_class_Типовой_sum', 'newbui_class_Элитный_sum']:
                empty_row[col] = np.nan
            results.append(empty_row)
            continue

        # --- РАСЧЕТ МЕТРИК ---
        row_result = {'fitness_id': idx}
        row_result['newbui_count'] = hits['LON'].count()
        row_result['newbui_LIVE_RMC_sum'] = hits['newbui_LIVE_RMC'].sum()
        row_result['newbui_LIVE_AREA_sum'] = hits['newbui_LIVE_RMC'].sum()
        row_result['newbui_class_Бизнес_sum'] = hits['newbui_LIVE_RMC'].sum()
        row_result['newbui_class_Комфорт_sum'] = hits['newbui_LIVE_RMC'].sum()
        row_result['newbui_class_Типовой_sum'] = hits['newbui_LIVE_RMC'].sum()
        row_result['newbui_class_Элитный_sum'] = hits['newbui_LIVE_RMC'].sum()
        row_result['newbui_year_build_mean'] = hits['newbui_year_build'].mean()
        
        results.append(row_result)
        
    except Exception as e:
        print(f"Ошибка на индексе {idx}: {e}")
        # В случае ошибки — записываем NaN для всех метрик
        empty_row = {'fitness_id': idx}
        for col in ['newbui_count', 'newbui_LIVE_RMC_sum', 'newbui_LIVE_AREA_sum','newbui_year_build_mean',
                        'newbui_class_Бизнес_sum', 'newbui_class_Комфорт_sum',
                        'newbui_class_Типовой_sum', 'newbui_class_Элитный_sum']:
            empty_row[col] = np.nan
        results.append(empty_row)

# --- СОХРАНЕНИЕ РЕЗУЛЬТАТА ---
res_df2 = pd.DataFrame(results)
res_df2.head()

Загружаем пешеходный граф Москвы...
НАЧАЛО ОБРАБОТКИ
Начинаем обработку 5029 объектов...
Обрабатываем объект 1/5029...
Обрабатываем объект 251/5029...
Обрабатываем объект 501/5029...
Обрабатываем объект 751/5029...
Обрабатываем объект 1001/5029...
Обрабатываем объект 1251/5029...
Обрабатываем объект 1501/5029...
Обрабатываем объект 1751/5029...
Обрабатываем объект 2001/5029...
Обрабатываем объект 2251/5029...
Обрабатываем объект 2501/5029...
Обрабатываем объект 2751/5029...
Обрабатываем объект 3001/5029...
Обрабатываем объект 3251/5029...
Обрабатываем объект 3501/5029...
Обрабатываем объект 3751/5029...
Обрабатываем объект 4001/5029...
Обрабатываем объект 4251/5029...
Обрабатываем объект 4501/5029...
Обрабатываем объект 4751/5029...
Обрабатываем объект 5001/5029...


,fitness_id,newbui_count,newbui_LIVE_RMC_sum,newbui_LIVE_AREA_sum,newbui_class_Бизнес_sum,newbui_class_Комфорт_sum,newbui_class_Типовой_sum,newbui_class_Элитный_sum,newbui_year_build_mean
0,0,238.0,79413.0,79413.0,79413.0,79413.0,79413.0,79413.0,2022.911765
1,1,216.0,96027.0,96027.0,96027.0,96027.0,96027.0,96027.0,2022.865741
2,2,193.0,57864.0,57864.0,57864.0,57864.0,57864.0,57864.0,2023.243523
3,3,200.0,92153.0,92153.0,92153.0,92153.0,92153.0,92153.0,2021.870000
4,4,307.0,107764.0,107764.0,107764.0,107764.0,107764.0,107764.0,2023.052117


In [77]:
res_df2 = pd.DataFrame(results)
print("Готово!")
# end = time.time()
print(f"Время: {end - start:.4f} сек")

lst_material = []
for col in list(res_df2.columns):
    if 'class' in col:
        lst_material.append(col)
res_df2['classes'] = res_df2[lst_material].sum(axis=1)

for col in list(res_df2.columns):
    if 'class' in col:
        res_df2[col] = res_df2[col]/res_df2['classes']

for col in list(res_df2.columns[1:]):
    old_name = col
    new_name = col + '_15drive'
    res_df2 = res_df2.rename(columns={old_name: new_name})
    
fitness_iso_newbui_drive = pd.concat([fitness,res_df2[1:-1]], axis=1)
fitness_iso_newbui_drive.to_csv('fitness_newbui_iso_15drive.csv', index = False)
fitness_iso_newbui_drive.to_excel('fitness_newbui_iso_15drive.xlsx', index = False)

fitness_iso_newbui_ped_drive = pd.concat([fitness_iso_pop,res_df2[1:-1]], axis=1)
fitness_iso_newbui_ped_drive.to_csv('fitness_newbui_iso_15ped_drive.csv', index = False)
fitness_iso_newbui_ped_drive.to_excel('fitness_newbui_iso_15ped_drive.xlsx', index = False)

Готово!
Время: 1523.1493 сек


In [78]:
res_df2

,fitness_id,newbui_count_15drive,newbui_LIVE_RMC_sum_15drive,newbui_LIVE_AREA_sum_15drive,newbui_class_Бизнес_sum_15drive,newbui_class_Комфорт_sum_15drive,newbui_class_Типовой_sum_15drive,newbui_class_Элитный_sum_15drive,newbui_year_build_mean_15drive,classes_15drive
0,0,238.0,79413.0,79413.0,0.25,0.25,0.25,0.25,2022.911765,1.0
1,1,216.0,96027.0,96027.0,0.25,0.25,0.25,0.25,2022.865741,1.0
2,2,193.0,57864.0,57864.0,0.25,0.25,0.25,0.25,2023.243523,1.0
3,3,200.0,92153.0,92153.0,0.25,0.25,0.25,0.25,2021.870000,1.0
4,4,307.0,107764.0,107764.0,0.25,0.25,0.25,0.25,2023.052117,1.0
...,...,...,...,...,...,...,...,...,...,...
5024,5024,155.0,90341.0,90341.0,0.25,0.25,0.25,0.25,2021.587097,1.0
5025,5025,180.0,87519.0,87519.0,0.25,0.25,0.25,0.25,2020.816667,1.0
5026,5026,42.0,18716.0,18716.0,0.25,0.25,0.25,0.25,2021.880952,1.0
5027,5027,39.0,15687.0,15687.0,0.25,0.25,0.25,0.25,2021.717949,1.0
